In [ ]:
# Q1. How many lesson pages
from gitsource import GithubRepositoryDataReader


reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(len(documents))

In [ ]:
# Q2. Indexing and searching
from minsearch import Index


index = Index(
    text_fields=["content"],
    keyword_fields=["filename"])

index.fit(documents)

search_results = index.search(
    query="How does the agentic loop keep calling the model until it stops?"
)

print(search_results[0]['filename'])

In [ ]:
# Q3. RAG
from homework_rag_helper import RAGBase

from openai import OpenAI


openai_client = OpenAI()

agent = RAGBase(
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

# I decided to keep the usage shown as part of the rag function call
# instead of complicating the return type.
response = agent.rag("How does the agentic loop keep calling the model until it stops?")

print(response)

ResponseUsage(input_tokens=7136, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=107, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7243)


In [ ]:
# Q4. Chunking
from gitsource import chunk_documents


chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

In [ ]:
# Q5. RAG with chunking
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"])

index.fit(chunks)


openai_client = OpenAI()

agent = RAGBase(
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

response = agent.rag("How does the agentic loop keep calling the model until it stops?")

ResponseUsage(input_tokens=2319, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=129, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2448)


In [18]:
# Q6. Turning it into an agent
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()


def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {'filename': 2.0, 'content': 1.0}
    # filter_dict = {'course': self.course}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        # filter_dict=filter_dict
    )


tools = Tools()
tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

print(result.cost)

-> Response received


-> Response received


CostInfo(input_cost=Decimal('0.00601425'), output_cost=Decimal('0.001674'), total_cost=Decimal('0.00768825'))
